# Enumerate the GovInfo Hearing Packages, 1995 to 2025

| Item | Description |
|---|---|
| Goal | List every congressional hearing package that GovInfo issued from 1995 to 2025, one row per package, before any transcript is downloaded. The list is the contract that D1.2 downloads against and that every later count is measured against. |
| Input | The GovInfo API, `/published` endpoint, collection `CHRG`. A free API key in `.env` in the D folder (template: `.env.example` beside it). |
| Output | In `Raw/manifest` of the D1 scratch folder: `packages_<year>.parquet`, one file per year; `packages_all.parquet`, all years in one table (36,378 rows with `packageId`, `dateIssued`, `lastModified`, `congress`, `docClass`, `title`, `packageLink`, `query_year`); `enumeration_summary.csv`, packages and API requests per year. |
| Run | Last run 2026-08-20. The cell outputs shown are from that run, which wrote to the old scratch location printed in them. Not rerun: the manifest is complete and every later stage was built on it. The files now sit in the Raw folder named above. |

## Step 1. Import packages

`requests` talks to the API. `pandas` holds the manifest.

In [ ]:
import os
import time
from pathlib import Path
from urllib.parse import parse_qs, urlparse

import pandas as pd
import requests

## Step 2. Define file locations

The project root is two levels above this notebook. The scratch root is the one absolute path. The manifest goes to `Raw/manifest` because it is data as received from the API.

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("D - Data Collection and Processing/D1 - GovInfo Congressional Hearings")
NOTEBOOK_NAME = "D1.1 - GovInfo - Enumerate Hearings"
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
MANIFEST_FOLDER = RAW_FOLDER / "manifest"  # one file per year, plus the combined manifest
ENV_FILE = PROJECT_FOLDER / "D - Data Collection and Processing" / ".env"  # holds GOVINFO_API_KEY

MANIFEST_FOLDER.mkdir(parents=True, exist_ok=True)

## Step 3. Set the collection window and read the API key

1995 is the first year the `CHRG` collection covers with any density. The key comes from the environment or from `.env`, never from the notebook text.

In [1]:
FIRST_YEAR = 1995
LAST_YEAR = 2025
COLLECTION = "CHRG"  # congressional hearings

API_KEY = os.environ.get("GOVINFO_API_KEY")
if not API_KEY and ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():  # one KEY=value per line
        name, _, value = line.strip().partition("=")
        if name == "GOVINFO_API_KEY" and value:
            API_KEY = value.strip().strip('"').strip("'")
assert API_KEY, f"GOVINFO_API_KEY is not set and {ENV_FILE} does not supply it; register a free key at https://api.data.gov/signup/"

print(f"pandas {pd.__version__}")
print(f"enumerating {COLLECTION} for {FIRST_YEAR}-{LAST_YEAR}")
print(f"manifest -> {MANIFEST_FOLDER}")

pandas 2.3.3
enumerating CHRG for 1995-2025
manifest -> /scratch2/acroh/projects/llm_method/data/raw/govinfo/manifest


## Step 4. Define one API request and one year's walk

A request carries the key in a header and waits a tenth of a second between calls. A transient failure is retried up to five times with a doubling wait, and any other error stops the notebook. A year's walk follows the `offsetMark` cursor to the last page, which stays correct while GovInfo keeps re-issuing packages. The walk then checks its record count against the `count` the API states and stops if they differ.

In [ ]:
BASE_URL = "https://api.govinfo.gov"
PAGE_SIZE = 1000  # the largest page GovInfo serves; fewer pages, fewer chances to be interrupted
SECONDS_BETWEEN_REQUESTS = 0.1  # 10 requests per second, a quarter of the published limit for a registered key
MAX_ATTEMPTS = 5  # per request; the wait doubles after each failure
TRANSIENT_STATUS = {408, 425, 429, 500, 502, 503, 504}  # retried; any other error code stops the notebook

session = requests.Session()
request_log = {"requests": 0, "retries": 0, "last_request": 0.0}


def govinfo_get(url, params):
    """One GET. Retries a transient failure, raises on anything else, never returns a failed response."""
    last_error = "no attempt made"
    for attempt in range(MAX_ATTEMPTS):
        wait = SECONDS_BETWEEN_REQUESTS - (time.monotonic() - request_log["last_request"])
        if wait > 0:
            time.sleep(wait)
        request_log["last_request"] = time.monotonic()
        request_log["requests"] += 1
        try:
            response = session.get(url, params=params, headers={"X-Api-Key": API_KEY}, timeout=120)
        except requests.RequestException as exc:  # connection reset, DNS failure, timeout
            last_error = f"{type(exc).__name__}: {exc}"
        else:
            if response.status_code < 400:
                return response
            if response.status_code not in TRANSIENT_STATUS:
                raise RuntimeError(f"HTTP {response.status_code} for {url}: {response.text[:200]}")
            last_error = f"HTTP {response.status_code}"
            retry_after = response.headers.get("Retry-After")
            if retry_after and retry_after.isdigit():
                time.sleep(int(retry_after))
                request_log["retries"] += 1
                continue
        request_log["retries"] += 1
        time.sleep(2.0 ** attempt)
    raise RuntimeError(f"{MAX_ATTEMPTS} attempts failed for {url}: {last_error}")


def list_published(year):
    """Every package GovInfo issued in one calendar year, following the offsetMark cursor to the end."""
    url = f"{BASE_URL}/published/{year}-01-01/{year}-12-31"
    # A cursor, not a position: GovInfo orders this list by lastModified and re-issues packages all the
    # time, so a position-based "offset" walk skips or repeats records while the list shifts underneath it.
    params = {"collection": COLLECTION, "pageSize": PAGE_SIZE, "offsetMark": "*"}
    records = []
    reported_count = None
    while True:
        payload = govinfo_get(url, params).json()
        if reported_count is None:
            reported_count = int(payload.get("count", 0))
        records.extend(payload.get("packages") or [])
        next_page = payload.get("nextPage")
        if not next_page:
            break
        params["offsetMark"] = parse_qs(urlparse(next_page).query)["offsetMark"][0]
    # The API states the total on every page. A walk that returns fewer records is an error, not a result.
    assert len(records) == reported_count, f"{year}: API reported {reported_count} packages but the walk returned {len(records)}"
    return records

## Step 5. Walk every year

One walk per calendar year, oldest first. A yearly slice keeps each walk short and matches the year folders of the raw tree. Reaching the end of this cell is itself the check that no year came back short.

In [2]:
frames = []
summary = []

for year in range(FIRST_YEAR, LAST_YEAR + 1):
    before = request_log["requests"]
    records = list_published(year)
    df = pd.DataFrame.from_records(records)
    df["query_year"] = year
    df.to_parquet(MANIFEST_FOLDER / f"packages_{year}.parquet", index=False)
    frames.append(df)
    summary.append({"year": year, "packages": len(df), "requests": request_log["requests"] - before})
    print(f"{year}  {len(df):>6,} packages")

manifest = pd.concat(frames, ignore_index=True)
summary = pd.DataFrame(summary)

1995     253 packages
1996     141 packages


1997     456 packages


1998     400 packages


1999     850 packages


2000     837 packages


2001   1,125 packages


... (20 outputs left out)

2022     965 packages


2023   1,320 packages


2024     896 packages


2025     744 packages


## Step 6. Save the manifest

One combined file sorted by issue date, plus the per-year request counts. D1.2 fetches exactly the packages listed here and records an outcome for each. A missing package is then a subtraction, not a guess.

In [3]:
manifest = manifest.sort_values(["dateIssued", "packageId"]).reset_index(drop=True)
manifest.to_parquet(MANIFEST_FOLDER / "packages_all.parquet", index=False)
summary.to_csv(MANIFEST_FOLDER / "enumeration_summary.csv", index=False)

print(f"{len(manifest):,} packages, {manifest['packageId'].nunique():,} distinct ids")
print(f"dateIssued spans {manifest['dateIssued'].min()} to {manifest['dateIssued'].max()}")
print(f"{request_log['requests']} API requests, {request_log['retries']} retries")
print()
print(manifest.head(3).to_string(index=False))

36,378 packages, 36,378 distinct ids
dateIssued spans 1995-01-01 to 2025-12-18
53 API requests, 0 retries

congress dateIssued docClass         lastModified         packageId                                                packageLink                                        title  query_year
     104 1995-01-01     HHRG 2026-06-01T20:18:38Z CHRG-104hhrg23219 https://api.govinfo.gov/packages/CHRG-104hhrg23219/summary Oversight Hearing on the Department of Labor        1995
     104 1995-01-01     HHRG 2026-01-27T13:50:35Z CHRG-104hhrg93189 https://api.govinfo.gov/packages/CHRG-104hhrg93189/summary         CALIFORNIA SPOTTED OWL RECOVERY PLAN        1995
     104 1995-01-05     HHRG 2025-03-07T16:22:29Z CHRG-104hhrg92037 https://api.govinfo.gov/packages/CHRG-104hhrg92037/summary               Contract with America-Overview        1995


## Step 7. Count packages by year and chamber

`docClass` separates House (`HHRG`), Senate (`SHRG`), and joint (`JHRG`) hearings. A year whose mix jumps is the first sign of a partial walk.

In [4]:
by_year = manifest.pivot_table(index="query_year", columns="docClass", values="packageId", aggfunc="count", fill_value=0)
by_year["total"] = by_year.sum(axis=1)
print(by_year.to_string())

docClass    GPO  HHRG  JHRG  SHRG  Y4  Y4_AG8_1  Y4_AG8_3  Y4_C73_8  Y4_EC7  Y4_F49  Y4_F49_20  Y4_F76_1  Y4_G74_7  Y4_IN8_16  Y4_J89_1  Y4_SE2  Y4_V64_3  total
query_year                                                                                                                                                      
1995          0   243     3     7   0         0         0         0       0       0          0         0         0          0         0       0         0    253
1996          0   139     2     0   0         0         0         0       0       0          0         0         0          0         0       0         0    141
1997          0   357     2    97   0         0         0         0       0       0          0         0         0          0         0       0         0    456
1998          0   279     2   119   0         0         0         0       0       0          0         0         0          0         0       0         0    400
1999          0   574     1   275 

## Step 8. Compare with the corpus the earlier pipeline used

The project's first summaries, topics, and relevance scores were built on 30,032 hearings. That collector could lose records without reporting it. The difference against this clean enumeration measures the gap.

In [5]:
PREVIOUS_CORPUS = 30_032  # distinct packageId the earlier summarization run covered
in_window = manifest.loc[manifest["query_year"].between(1995, 2022)]

print(f"clean enumeration, 1995-2022 : {len(in_window):>7,}")
print(f"earlier pipeline corpus      : {PREVIOUS_CORPUS:>7,}")
print(f"difference                   : {len(in_window) - PREVIOUS_CORPUS:>7,} "
      f"({(len(in_window) - PREVIOUS_CORPUS) / len(in_window):.1%} of the corpus)")
print()
print(f"beyond the earlier window (2023-2025): {len(manifest) - len(in_window):,}")

clean enumeration, 1995-2022 :  33,418
earlier pipeline corpus      :  30,032
difference                   :   3,386 (10.1% of the corpus)

beyond the earlier window (2023-2025): 2,960


## Step 9. Check the manifest

Four checks, each one stopping the notebook if it fails.

- Every `packageId` is distinct.
- Every year in the window returned records.
- Every `dateIssued` falls in the year it was queried under.
- The fields D1.2 builds URLs and file names from are never null.

In [6]:
dupes = manifest["packageId"].duplicated().sum()
assert dupes == 0, f"{dupes} duplicate packageId: the cursor walk revisited a page"
print(f"1) {len(manifest):,} rows, all packageId distinct")

missing_years = sorted(set(range(FIRST_YEAR, LAST_YEAR + 1)) - set(manifest["query_year"]))
assert not missing_years, f"years returned no records at all: {missing_years}"
print(f"2) all {LAST_YEAR - FIRST_YEAR + 1} years present, none empty")

issued_year = pd.to_datetime(manifest["dateIssued"], format="mixed", utc=True).dt.year
off_window = int((issued_year != manifest["query_year"]).sum())
assert off_window == 0, f"{off_window} records fall outside the year they were queried under"
print("3) every dateIssued falls inside its query year")

for column in ("packageId", "dateIssued", "docClass"):
    n_null = int(manifest[column].isna().sum())
    assert n_null == 0, f"{n_null} null {column}: D1.2 builds URLs and file names from these"
print("4) packageId, dateIssued, docClass all non-null")

print()
print(f"manifest ready: {MANIFEST_FOLDER / 'packages_all.parquet'}")

1) 36,378 rows, all packageId distinct
2) all 31 years present, none empty
3) every dateIssued falls inside its query year
4) packageId, dateIssued, docClass all non-null

manifest ready: /scratch2/acroh/projects/llm_method/data/raw/govinfo/manifest/packages_all.parquet
